# Phase 1B — Prithvi-EO-2.0-300M Linear: Aggregate-Label Benchmark
**Runtime:** Google Colab — **GPU (T4 minimum, A100 preferred)**

**Purpose:** Aggregate-label ablation of NB07 (Prithvi-300M linear probe).  
Strategy: `linear` — entire encoder frozen, only the regression head is trained.
- Encoder: `prithvi_eo_v2_300` (frozen) via TerraTorch `BACKBONE_REGISTRY`
- Head: `LayerNorm(1024) → Dropout(0.1) → Linear(1024, 1)` (scalar output)
- Loss: MSE

Two models are trained sequentially — one per aggregate target:

| Target | Formula | Interpretation |
|---|---|---|
| `coverage_fraction` | n_ookla_tiles_in_chip / total_possible_zoom16_tiles | Spatial coverage |
| `log_mean_tests` | log(1 + total_tests / n_subtiles) | Test density per sub-tile |

Predictions are binarised with a val-optimal threshold and evaluated against the binary
ground-truth labels on the same Northeast India geographic hold-out as NB07.

**Inputs:**
- `data/raw/patches_2019/` — 6,970 GeoTIFF patches (224×224×6 HLS bands)
- `data/processed/sampled_patches.csv` — patch metadata + binary labels
- `data/raw/ookla_india_2019_Q1.csv` — raw Ookla sub-tile data (from Google Drive if needed)

## Step 0: Environment Setup

In [ ]:
# ============================================================
# Cell 0.1: Install dependencies
# TerraTorch MUST be installed first — restart runtime after this cell.
# ============================================================
# Pin numpy BEFORE terratorch so it cannot be upgraded to 2.x
# (prevents "numpy.dtype size changed" ABI mismatch with pandas/scipy)
!pip install -q "numpy==1.26.4"
!pip install -q terratorch
!pip install -q rasterio scikit-learn geopandas matplotlib seaborn scipy pyarrow joblib

# Restart runtime so all C extensions load against the pinned numpy
import os
print("Restarting runtime to apply numpy pin — re-run from Step 0.2 after restart.")
os.kill(os.getpid(), 9)


In [ ]:
# ============================================================
# Cell 0.2: Clone repo
# ============================================================
import os

REPO = 'internet-connectivity-prediction'
if not os.getcwd().endswith(REPO):
    if os.path.exists(REPO):
        %cd {REPO}
    else:
        !git clone https://github.com/tatyana21111/{REPO}.git
        %cd {REPO}
!git pull

In [ ]:
# ============================================================
# Cell 0.3: Sync patches from Google Drive
# ============================================================
from google.colab import drive
import shutil
from pathlib import Path

drive.mount('/content/drive', force_remount=True)

PATCH_DIR = 'data/raw/patches_2019'
Path(PATCH_DIR).mkdir(parents=True, exist_ok=True)

local_count = len(list(Path(PATCH_DIR).glob('*.tif')))
if local_count < 6900:
    print('Syncing patches from Google Drive ...')
    drive_dir = '/content/drive/MyDrive/patches_2019'
    for f in Path(drive_dir).glob('*.tif'):
        dst = Path(PATCH_DIR) / f.name
        if not dst.exists():
            shutil.copy(f, dst)
    local_count = len(list(Path(PATCH_DIR).glob('*.tif')))
print(f'Patches available: {local_count:,}')

## Step 1: Imports & Constants

In [ ]:
import random, numpy as np, torch
random.seed(42); np.random.seed(42)
torch.manual_seed(42); torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True; torch.use_deterministic_algorithms(True)

import pandas as pd
import geopandas as gpd
import rasterio
import torch.nn as nn
import pytorch_lightning as pl
import matplotlib.pyplot as plt
import warnings
import logging
import json
from pathlib import Path
from shapely.geometry import box
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    average_precision_score, precision_recall_curve,
    roc_auc_score, f1_score, precision_score,
    recall_score, accuracy_score, confusion_matrix,
    classification_report, mean_absolute_error, mean_squared_error,
)
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
logging.getLogger('rasterio').setLevel(logging.ERROR)

HLS_MEANS = [0.14245495, 0.13921481, 0.12434631, 0.31420089, 0.20743526, 0.12046503]
HLS_STDS  = [0.04036231, 0.04186983, 0.05267646, 0.08222210, 0.06834774, 0.05294205]
SCALE     = 0.0001

PATCH_DIR   = 'data/raw/patches_2019'
RESULTS_DIR = 'outputs/results'
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
Path('outputs/figures').mkdir(parents=True, exist_ok=True)
Path('outputs/models').mkdir(parents=True, exist_ok=True)
Path('checkpoints').mkdir(parents=True, exist_ok=True)

# HLS chip: 224 px × 30 m = 6720 m per side
PATCH_SIZE_M   = 6720.0
PATCH_SIZE_DEG = PATCH_SIZE_M / 111_320.0
EARTH_CIRC_M   = 40_075_016.686
ZOOM16_SIDE_EQ = EARTH_CIRC_M / (2 ** 16)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

## Step 2: Load Raw Ookla Sub-Tile CSV

In [ ]:
OOKLA_LOCAL = Path('data/raw/ookla_india_2019_Q1.csv')
OOKLA_DRIVE = '/content/drive/MyDrive/gee_exports/ookla_india_2019_Q1.csv'

if not OOKLA_LOCAL.exists():
    print(f'Copying Ookla CSV from Drive: {OOKLA_DRIVE}')
    shutil.copy(OOKLA_DRIVE, OOKLA_LOCAL)

ookla_raw = pd.read_csv(OOKLA_LOCAL)
print(f'Ookla tiles loaded: {len(ookla_raw):,}  columns: {list(ookla_raw.columns)}')

In [ ]:
def parse_geo(geo_str):
    return gpd.GeoDataFrame.from_features(
        [{'type': 'Feature', 'geometry': json.loads(geo_str), 'properties': {}}]
    ).geometry[0]

ookla_raw['geometry'] = ookla_raw['.geo'].apply(parse_geo)
ookla_gdf = gpd.GeoDataFrame(ookla_raw, geometry='geometry', crs='EPSG:4326')
print(f'Ookla GeoDataFrame: {len(ookla_gdf):,} tiles')

## Step 3: Build Patch Polygons, Spatial Join & Compute Targets

In [ ]:
sampled = pd.read_csv('data/processed/sampled_patches.csv')
available = set(f.stem for f in Path(PATCH_DIR).glob('*.tif'))
sampled   = sampled[sampled['patch_id'].isin(available)].reset_index(drop=True)
sampled['has_coverage'] = (sampled['label_confidence'] == 'confirmed_positive').astype(int)

print(f'Patches available: {len(sampled):,}')
print(f'Binary positive rate: {sampled["has_coverage"].mean():.2%}')

half = PATCH_SIZE_DEG / 2
patch_gdf = gpd.GeoDataFrame(
    sampled[['patch_id', 'lon', 'lat']],
    geometry=[box(lon - half, lat - half, lon + half, lat + half)
              for lon, lat in zip(sampled['lon'], sampled['lat'])],
    crs='EPSG:4326'
)
print(f'Patch polygons built: {len(patch_gdf):,}')

In [ ]:
print('Running spatial join ...')
joined = gpd.sjoin(
    ookla_gdf[['tests', 'geometry']],
    patch_gdf[['patch_id', 'geometry']],
    how='inner',
    predicate='intersects',
)

ookla_per_patch = (
    joined.groupby('patch_id')
    .agg(
        total_tests = ('tests', 'sum'),
        n_subtiles  = ('tests', 'count'),
        mean_tests  = ('tests', 'mean'),
    )
    .reset_index()
)

sampled = sampled.merge(ookla_per_patch, on='patch_id', how='left')
for col in ['total_tests', 'n_subtiles', 'mean_tests']:
    sampled[col].fillna(0, inplace=True)

print(f'Patches with >=1 Ookla sub-tile: {(sampled["n_subtiles"] > 0).sum():,}')
print(f'Patches with 0  Ookla sub-tiles: {(sampled["n_subtiles"] == 0).sum():,}')

In [ ]:
def total_possible_tiles(lat, patch_m=PATCH_SIZE_M, side_eq=ZOOM16_SIDE_EQ):
    tile_side = side_eq * np.cos(np.radians(lat))
    return (patch_m / tile_side) ** 2

sampled['total_possible_tiles'] = sampled['lat'].apply(total_possible_tiles)
sampled['coverage_fraction'] = (
    sampled['n_subtiles'] / sampled['total_possible_tiles']
).clip(0, 1)
sampled['log_mean_tests'] = np.log1p(sampled['mean_tests'])

print('=== Aggregate targets ===')
print('coverage_fraction:')
print(sampled['coverage_fraction'].describe().round(4))
print('\nlog_mean_tests:')
print(sampled['log_mean_tests'].describe().round(4))

## Step 4: Train / Val / Test Split (same geographic holdout as NB07)

In [ ]:
northeast_mask = (sampled['lon'] > 89.0) & (sampled['lat'] > 21.0)
test_df   = sampled[northeast_mask].copy().reset_index(drop=True)
train_val = sampled[~northeast_mask].copy()

train_df, val_df = train_test_split(
    train_val, test_size=0.2,
    stratify=train_val['has_coverage'],
    random_state=42
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test (NE): {len(test_df):,}')
print(f'Test binary positive rate   : {test_df["has_coverage"].mean():.2%}')
print(f'Train coverage_fraction mean: {train_df["coverage_fraction"].mean():.4f}')
print(f'Test  coverage_fraction mean: {test_df["coverage_fraction"].mean():.4f}')

## Step 5: Load Cached Prithvi Embeddings + Dataset
Load frozen 1024-d Prithvi embeddings from `outputs/features/` (extracted by NB16).
If the cache is missing, extract and save them inline — this only happens once.
Subsequent epochs train only the lightweight regression head on pre-extracted embeddings,
which is much faster and ensures consistency across notebooks.

In [ ]:
OUTPUT_FEATURES = Path('outputs/features')
OUTPUT_FEATURES.mkdir(parents=True, exist_ok=True)


def load_or_extract_embeddings(split_name, df):
    """Load cached frozen Prithvi embeddings, or extract + cache them."""
    path = OUTPUT_FEATURES / f'prithvi_frozen_embeds_{split_name}.npz'
    if path.exists():
        data = np.load(path)
        X = data['X'].astype(np.float32)
        print(f'  Loaded cached {split_name}: {X.shape}')
        return X

    print(f'  Cache miss for {split_name} — extracting from scratch ...')
    from terratorch.registry import BACKBONE_REGISTRY

    encoder = BACKBONE_REGISTRY.build('prithvi_eo_v2_300', pretrained=True).to(DEVICE).eval()
    for p in encoder.parameters():
        p.requires_grad = False

    class PrithviPatchDataset(Dataset):
        def __init__(self, df, patch_dir, n_bands=6):
            self.df = df.reset_index(drop=True)
            self.patch_dir = patch_dir
            self.n_bands = n_bands
        def __len__(self):
            return len(self.df)
        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            with rasterio.open(f"{self.patch_dir}/{row['patch_id']}.tif") as src:
                img = src.read(list(range(1, self.n_bands + 1))).astype(np.float32)
            img *= SCALE
            for b in range(self.n_bands):
                img[b] = (img[b] - HLS_MEANS[b]) / HLS_STDS[b]
            img = np.nan_to_num(img, nan=0.0)
            return torch.from_numpy(img).unsqueeze(1), row['patch_id']

    ds = PrithviPatchDataset(df, PATCH_DIR)
    loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

    all_embs = []
    with torch.no_grad():
        for x, _ in tqdm(loader, desc=f'Extracting {split_name}'):
            x = x.to(DEVICE, dtype=torch.float32)
            feats = encoder(x)
            pooled = feats[-1].mean(dim=1)
            all_embs.append(pooled.cpu().numpy().astype(np.float32))

    X = np.concatenate(all_embs, axis=0)
    np.savez_compressed(path, X=X, patch_id=df['patch_id'].to_numpy())
    print(f'  Extracted and cached {split_name}: {X.shape}')

    del encoder
    torch.cuda.empty_cache()
    return X


print('Loading frozen Prithvi embeddings (1024-d) ...')
X_train_emb = load_or_extract_embeddings('train', train_df)
X_val_emb   = load_or_extract_embeddings('val',   val_df)
X_test_emb  = load_or_extract_embeddings('test',  test_df)
print(f'\nTrain: {X_train_emb.shape}  |  Val: {X_val_emb.shape}  |  Test: {X_test_emb.shape}')


class EmbeddingDataset(Dataset):
    """Simple dataset wrapping pre-extracted embeddings + scalar targets."""
    def __init__(self, embeddings, targets):
        self.X = torch.from_numpy(embeddings)
        self.y = torch.tensor(targets, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


_ds = EmbeddingDataset(X_train_emb, train_df['coverage_fraction'].values)
_x, _y = _ds[0]
print(f'Embedding shape: {_x.shape}  |  Target: {_y.item():.4f}')
del _ds, _x, _y

## Step 6: PrithviRegressor — Head-Only (Embeddings Pre-Extracted)
Since embeddings are pre-computed and cached, the model only contains the regression head.
No encoder on GPU during training → faster training and lower memory usage.

In [ ]:
class PrithviRegressor(pl.LightningModule):
    """
    Regression head trained on frozen Prithvi embeddings (1024-d).
    Head: LayerNorm(1024) → Dropout(p) → Linear(1024, 1)
    Loss: MSE
    """

    def __init__(self, target_col, lr=1e-3, weight_decay=1e-3, dropout=0.1):
        super().__init__()
        self.save_hyperparameters()

        self.head = nn.Sequential(
            nn.LayerNorm(1024),
            nn.Dropout(dropout),
            nn.Linear(1024, 1),
        )
        self.loss_fn = nn.MSELoss()

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'Target: {target_col}  |  Trainable params: {trainable:,}')

    def forward(self, x):
        """x: (B, 1024) pre-extracted embeddings → scalar predictions (B,)"""
        return self.head(x).squeeze(1)

    def training_step(self, batch, batch_idx):
        x, y   = batch
        preds  = self(x)
        loss   = self.loss_fn(preds, y)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y  = batch
        preds = self(x)
        loss  = self.loss_fn(preds, y)
        self.log('val_loss', loss, prog_bar=True, sync_dist=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay,
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=10
        )
        return {
            'optimizer': optimizer,
            'lr_scheduler': {'scheduler': scheduler, 'monitor': 'val_loss'},
        }

## Step 7: Training — Both Targets (Fixed Conservative Defaults)

Due to project time constraints, the linear head is trained with a small set of
conservative default hyperparameters chosen to prioritize stability under
geographic distribution shift.

| Param | Value |
|---|---|
| `learning_rate` | 1e-3 |
| `weight_decay` | 1e-3 |
| `dropout` | 0.1 |
| `batch_size` | 32 |
| `max_epochs` | 100 |
| `early_stopping_patience` | 15 |
| `optimizer` | AdamW |
| `loss` | MSELoss |

In [ ]:
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint


class MetricTracker(pl.Callback):
    """Records val_loss (and train_loss) at the end of every epoch."""
    def __init__(self):
        self.train_loss = []
        self.val_loss   = []

    def on_train_epoch_end(self, trainer, pl_module):
        v = trainer.callback_metrics.get('train_loss')
        self.train_loss.append(float(v) if v is not None else float('nan'))

    def on_validation_epoch_end(self, trainer, pl_module):
        v = trainer.callback_metrics.get('val_loss')
        self.val_loss.append(float(v) if v is not None else float('nan'))


# ── Fixed conservative defaults ──────────────────────────────
FIXED_CFG = {
    'lr':           1e-3,
    'weight_decay': 1e-3,
    'dropout':      0.1,
    'batch_size':   32,
    'max_epochs':   100,
    'patience':     15,
}

print(f'Fixed config: {FIXED_CFG}')

# ── Training per target ─────────────────────────────────────
best_ckpt_paths = {}
val_histories   = {}

for TARGET_NAME in ['coverage_fraction', 'log_mean_tests']:
    print(f'\n{"="*70}')
    print(f'  TRAINING — {TARGET_NAME}')
    print(f'{"="*70}')

    train_loader = DataLoader(
        EmbeddingDataset(X_train_emb, train_df[TARGET_NAME].values),
        batch_size=FIXED_CFG['batch_size'], shuffle=True,
        num_workers=0, pin_memory=True,
    )
    val_loader = DataLoader(
        EmbeddingDataset(X_val_emb, val_df[TARGET_NAME].values),
        batch_size=FIXED_CFG['batch_size'] * 2, shuffle=False,
        num_workers=0, pin_memory=True,
    )

    model = PrithviRegressor(
        target_col=TARGET_NAME,
        lr=FIXED_CFG['lr'],
        weight_decay=FIXED_CFG['weight_decay'],
        dropout=FIXED_CFG['dropout'],
    )
    metric_tracker = MetricTracker()
    early_stop = EarlyStopping(
        monitor='val_loss', patience=FIXED_CFG['patience'],
        mode='min', verbose=True,
    )
    checkpoint_cb = ModelCheckpoint(
        dirpath='checkpoints',
        filename=f'prithvi_linear_agg_{TARGET_NAME}_' + '{epoch:02d}_{val_loss:.4f}',
        monitor='val_loss', mode='min', save_top_k=1, verbose=True,
    )

    trainer = Trainer(
        max_epochs=FIXED_CFG['max_epochs'],
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        callbacks=[metric_tracker, early_stop, checkpoint_cb],
        log_every_n_steps=10,
        enable_progress_bar=True,
    )
    trainer.fit(model, train_loader, val_loader)

    best_ckpt_paths[TARGET_NAME] = checkpoint_cb.best_model_path
    val_histories[TARGET_NAME]   = metric_tracker.val_loss
    print(f'Best checkpoint: {checkpoint_cb.best_model_path}')
    print(f'Best val loss  : {checkpoint_cb.best_model_score:.6f}')
    print(f'Trained epochs : {len(metric_tracker.val_loss)}')

    del model, trainer, train_loader, val_loader
    torch.cuda.empty_cache()

print('\n All training complete.')
print('Checkpoints:', best_ckpt_paths)

## Step 8: Evaluation

Per target, four diagnostic plot groups are produced:

| Figure | Content | Question answered |
|---|---|---|
| 1 | Validation loss over epoch | Did training converge? When did it stop? |
| 2 | Predicted vs actual scatter + MAE/RMSE bar | How well does the model track the continuous target? |
| 3 | Residuals vs true value | Where does the model struggle (low vs high density chips)? |
| 4 | Spatial prediction map (lon/lat) | Does the model learn coherent spatial patterns? |
| 5 | Confusion matrix + PR curve | Does the continuous score help on the binary connectivity task? |

Self-contained: resolves checkpoints from `checkpoints/` directory.

In [ ]:
# ============================================================
# Cell 8.1: Checkpoint resolution helper
# Priority 1: best_ckpt_paths dict from training above (in-session)
# Priority 2: scan checkpoints/ directory for most recent matching file
# ============================================================
def resolve_checkpoint(target_name):
    # Priority 1: in-session training result
    if 'best_ckpt_paths' in globals() and target_name in best_ckpt_paths:
        p = best_ckpt_paths[target_name]
        if Path(p).exists():
            print(f'[{target_name}] Using in-session checkpoint: {p}')
            return p
    # Priority 2: scan checkpoints/
    candidates = sorted(
        Path('checkpoints').glob(f'prithvi_linear_agg_{target_name}_*.ckpt'),
        key=lambda p: p.stat().st_mtime, reverse=True
    )
    if candidates:
        print(f'[{target_name}] Found checkpoint: {candidates[0]}')
        return str(candidates[0])
    raise FileNotFoundError(
        f'No checkpoint found for target "{target_name}". Run training (Step 7) first.'
    )

In [ ]:
# ============================================================
# Cell 8.2: Inference helper — uses cached embeddings
# ============================================================
def run_inference(model, embeddings, targets, batch_size=256):
    """Run head-only inference on pre-extracted embeddings."""
    dataset = EmbeddingDataset(embeddings, targets)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    preds_list = []
    model.eval()
    with torch.no_grad():
        for x, _ in loader:
            preds_list.extend(model(x.to(DEVICE)).cpu().numpy())
    return np.array(preds_list), targets

In [ ]:
# ============================================================
# Cell 8.3: Evaluate both targets
# ============================================================
from scipy.stats import binned_statistic

all_metrics = {}

for TARGET_NAME in ['coverage_fraction', 'log_mean_tests']:
    print(f'\n{"="*60}')
    print(f'Evaluating: {TARGET_NAME}')
    print(f'{"="*60}')

    ckpt_path = resolve_checkpoint(TARGET_NAME)
    reg_model = PrithviRegressor.load_from_checkpoint(ckpt_path).to(DEVICE).eval()
    print('Model loaded.')

    # ── Figure 1: Validation loss over epoch ─────────────────
    hist = val_histories.get(TARGET_NAME, []) if 'val_histories' in globals() else []
    if hist:
        fig, ax = plt.subplots(figsize=(8, 4))
        epochs     = list(range(1, len(hist) + 1))
        best_epoch = int(np.argmin(hist)) + 1
        ax.plot(epochs, hist, lw=2, color='#4C72B0', label='Val loss (MSE)')
        ax.axvline(best_epoch, linestyle='--', color='red', alpha=0.8,
                   label=f'Best epoch {best_epoch}  (loss = {min(hist):.4f})')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Val MSE Loss')
        ax.set_title(f'Validation Loss over Epoch — {TARGET_NAME}')
        ax.legend(); plt.tight_layout()
        plt.savefig(f'outputs/figures/prithvi_linear_agg_{TARGET_NAME}_val_loss.png',
                    dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print('(val_histories not available — run training first to see learning curve)')

    # ── Val inference → optimal binarisation threshold ───────
    val_preds, _ = run_inference(reg_model, X_val_emb, val_df[TARGET_NAME].values)
    y_val_bin    = val_df['has_coverage'].values

    prec_v, rec_v, thr_v = precision_recall_curve(y_val_bin, val_preds)
    f1_v    = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-8)
    opt_thr = float(thr_v[np.argmax(f1_v)])
    print(f'Val-optimal threshold: {opt_thr:.4f}')

    # ── Test inference ────────────────────────────────────────
    test_preds, _ = run_inference(reg_model, X_test_emb, test_df[TARGET_NAME].values)
    y_test_bin    = test_df['has_coverage'].values
    y_test_cont   = test_df[TARGET_NAME].values
    y_pred_bin    = (test_preds >= opt_thr).astype(int)
    residuals     = test_preds - y_test_cont

    # ── PRIMARY: regression quality ──────────────────────────
    mae    = mean_absolute_error(y_test_cont, test_preds)
    rmse   = np.sqrt(mean_squared_error(y_test_cont, test_preds))
    rho, _ = spearmanr(y_test_cont, test_preds)

    print('\n--- PRIMARY: Regression quality ---')
    print(f'  MAE          : {mae:.4f}')
    print(f'  RMSE         : {rmse:.4f}')
    print(f'  Spearman rho : {rho:.4f}')

    # ── SECONDARY: binary connectivity task ──────────────────
    pr_auc   = average_precision_score(y_test_bin, test_preds)
    auc      = roc_auc_score(y_test_bin, test_preds)
    f1_opt   = f1_score(y_test_bin, y_pred_bin)
    prec_opt = precision_score(y_test_bin, y_pred_bin, zero_division=0)
    rec_opt  = recall_score(y_test_bin, y_pred_bin)
    acc_opt  = accuracy_score(y_test_bin, y_pred_bin)

    print('\n--- SECONDARY: Binary connectivity task ---')
    print(f'  PR-AUC        : {pr_auc:.4f}   (primary for imbalanced data)')
    print(f'  ROC-AUC       : {auc:.4f}')
    print(f'  F1 @ thr={opt_thr:.3f} : {f1_opt:.4f}')
    print(f'  Precision     : {prec_opt:.4f}  |  Recall: {rec_opt:.4f}  |  Acc: {acc_opt:.4f}')
    print()
    print(classification_report(y_test_bin, y_pred_bin,
                                 target_names=['No Coverage', 'Has Coverage']))

    # ── Figure 2: Predicted vs actual + Residual histogram ───
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f'Regression Quality — {TARGET_NAME}', fontsize=13, fontweight='bold')

    axes[0].scatter(y_test_cont, test_preds, alpha=0.35, s=12, color='#4C72B0')
    mn = min(y_test_cont.min(), test_preds.min())
    mx = max(y_test_cont.max(), test_preds.max())
    axes[0].plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='45° reference')
    axes[0].set_xlabel(f'Actual {TARGET_NAME}')
    axes[0].set_ylabel(f'Predicted {TARGET_NAME}')
    axes[0].set_title(
        f'Predicted vs Actual\nMAE={mae:.3f}  RMSE={rmse:.3f}  Spearman ρ={rho:.3f}')
    axes[0].legend(fontsize=9)

    axes[1].hist(residuals, bins=40, color='#4C72B0', edgecolor='white', alpha=0.8)
    axes[1].axvline(0, color='red', linestyle='--', lw=1.5, label='Zero')
    axes[1].axvline(residuals.mean(), color='orange', linestyle='-', lw=1.5,
                    label=f'Mean = {residuals.mean():.3f}')
    axes[1].set_xlabel('Residual (predicted − true)')
    axes[1].set_ylabel('Count')
    axes[1].set_title(f'Residual Distribution  (std = {residuals.std():.3f})')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(f'outputs/figures/prithvi_linear_agg_{TARGET_NAME}_regression.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    # ── Figure 3: Residuals vs true value ────────────────────
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(y_test_cont, residuals, alpha=0.35, s=12, color='#4C72B0')
    ax.axhline(0, color='red', linestyle='--', lw=1.5, label='Zero residual')

    try:
        bin_means, bin_edges, _ = binned_statistic(
            y_test_cont, residuals, statistic='mean', bins=10)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        ax.plot(bin_centers, bin_means, 'o-', color='orange', lw=2,
                ms=6, label='Bin mean residual')
    except Exception:
        pass

    ax.set_xlabel(f'True {TARGET_NAME}')
    ax.set_ylabel('Residual  (predicted − true)')
    ax.set_title(
        f'Residuals vs True Value — {TARGET_NAME}\n'
        f'Upward slope = model undershoots high-density chips; '
        f'downward = overshoots')
    ax.legend(); plt.tight_layout()
    plt.savefig(f'outputs/figures/prithvi_linear_agg_{TARGET_NAME}_residuals.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    # ── Figure 4: Spatial prediction map ─────────────────────
    lons = test_df['lon'].values
    lats = test_df['lat'].values
    res_abs_max = np.percentile(np.abs(residuals), 95)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'Spatial Prediction Map — {TARGET_NAME}\nNE India test set',
                 fontsize=13, fontweight='bold')

    sc0 = axes[0].scatter(lons, lats, c=test_preds,
                          cmap='RdYlGn', s=18, alpha=0.75)
    plt.colorbar(sc0, ax=axes[0], label=f'Predicted {TARGET_NAME}')
    axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
    axes[0].set_title('Predicted value')

    sc1 = axes[1].scatter(lons, lats, c=residuals,
                          cmap='RdBu_r', s=18, alpha=0.75,
                          vmin=-res_abs_max, vmax=res_abs_max)
    plt.colorbar(sc1, ax=axes[1], label='Residual (predicted − true)')
    axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')
    axes[1].set_title('Residuals  (blue = under-predicted, red = over-predicted)')

    plt.tight_layout()
    plt.savefig(f'outputs/figures/prithvi_linear_agg_{TARGET_NAME}_spatial.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    # ── Figure 5: Binary connectivity task ───────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f'Binary Connectivity Task — {TARGET_NAME}', fontsize=13, fontweight='bold')

    cm = confusion_matrix(y_test_bin, y_pred_bin)
    im = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
    fig.colorbar(im, ax=axes[0])
    axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['No Cov.', 'Has Cov.'])
    axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(['No Cov.', 'Has Cov.'])
    for r in range(2):
        for c in range(2):
            axes[0].text(c, r, str(cm[r, c]), ha='center', va='center',
                         color='white' if cm[r, c] > cm.max() / 2 else 'black', fontsize=14)
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
    axes[0].set_title(f'Confusion Matrix  (thr = {opt_thr:.3f})')

    prec_t, rec_t, _ = precision_recall_curve(y_test_bin, test_preds)
    axes[1].plot(rec_t, prec_t, lw=2, label=f'PR-AUC = {pr_auc:.3f}')
    axes[1].axhline(y_test_bin.mean(), linestyle='--', color='grey',
                    label=f'Baseline ({y_test_bin.mean():.3f})')
    axes[1].scatter([rec_opt], [prec_opt], marker='*', s=200, color='red', zorder=5,
                    label=f'Val-opt thr = {opt_thr:.3f}')
    axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
    axes[1].set_title('Precision-Recall Curve')
    axes[1].legend(loc='upper right'); axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1])

    plt.tight_layout()
    plt.savefig(f'outputs/figures/prithvi_linear_agg_{TARGET_NAME}_binary.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    all_metrics[TARGET_NAME] = {
        'model':         f'Prithvi_300M_linear_agg_{TARGET_NAME}',
        'target':        TARGET_NAME,
        'mae':           mae,
        'rmse':          rmse,
        'spearman_rho':  rho,
        'pr_auc':        pr_auc,
        'roc_auc':       auc,
        'f1_opt':        f1_opt,
        'opt_threshold': opt_thr,
        'precision_opt': prec_opt,
        'recall_opt':    rec_opt,
        'accuracy_opt':  acc_opt,
        'n_train':       len(train_df) + len(val_df),
        'n_test':        len(test_df),
    }

    del reg_model
    torch.cuda.empty_cache()

## Step 9: Summary — Regression Quality + Binary Task Comparison

In [ ]:
m_cov = all_metrics['coverage_fraction']
m_lmt = all_metrics['log_mean_tests']

# ── Primary: regression quality table ────────────────────────
reg_df = pd.DataFrame([
    {'Target': 'coverage_fraction', 'MAE': m_cov['mae'], 'RMSE': m_cov['rmse'],
     'Spearman_rho': m_cov['spearman_rho']},
    {'Target': 'log_mean_tests',    'MAE': m_lmt['mae'], 'RMSE': m_lmt['rmse'],
     'Spearman_rho': m_lmt['spearman_rho']},
])
print('=== PRIMARY: Regression quality ===')
print(reg_df.to_string(index=False))

# ── Secondary: binary task comparison with NB07 baseline ─────
nb07_path = Path('outputs/results/prithvi_linear_ft_metrics.csv')
if nb07_path.exists():
    base      = pd.read_csv(nb07_path).iloc[0]
    base_pr   = base.get('pr_auc',   float('nan'))
    base_auc  = base.get('roc_auc',  float('nan'))
    base_f1   = base.get('f1_opt',   float('nan'))
else:
    base_pr = base_auc = base_f1 = float('nan')
    print('NB07 baseline not found — run NB07 first for full comparison.')

bin_df = pd.DataFrame([
    {'Model': 'NB07 binary label',         'PR-AUC': base_pr,         'ROC-AUC': base_auc,         'F1': base_f1},
    {'Model': 'NB14 coverage_fraction',    'PR-AUC': m_cov['pr_auc'], 'ROC-AUC': m_cov['roc_auc'], 'F1': m_cov['f1_opt']},
    {'Model': 'NB14 log_mean_tests',       'PR-AUC': m_lmt['pr_auc'], 'ROC-AUC': m_lmt['roc_auc'], 'F1': m_lmt['f1_opt']},
])
print('\n=== SECONDARY: Binary connectivity task ===')
print(bin_df.to_string(index=False))

# ── Figure: two-panel summary ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Prithvi-300M Linear — Aggregate-Label Benchmark\nNE India test set',
             fontsize=13, fontweight='bold')

# Left panel: regression metrics (MAE + RMSE bars, Spearman ρ annotated)
targets   = ['coverage_fraction', 'log_mean_tests']
mae_vals  = [m_cov['mae'],  m_lmt['mae']]
rmse_vals = [m_cov['rmse'], m_lmt['rmse']]
rho_vals  = [m_cov['spearman_rho'], m_lmt['spearman_rho']]
x, w = np.arange(len(targets)), 0.35
axes[0].bar(x - w/2, mae_vals,  w, label='MAE',  color='#4C72B0')
axes[0].bar(x + w/2, rmse_vals, w, label='RMSE', color='#DD8452')
for i, (mv, rv, rho_v) in enumerate(zip(mae_vals, rmse_vals, rho_vals)):
    axes[0].text(i - w/2, mv  + max(mae_vals + rmse_vals) * 0.02,
                 f'{mv:.3f}', ha='center', va='bottom', fontsize=9)
    axes[0].text(i + w/2, rv  + max(mae_vals + rmse_vals) * 0.02,
                 f'{rv:.3f}', ha='center', va='bottom', fontsize=9)
    axes[0].text(i, max(mv, rv) * 1.18,
                 f'ρ = {rho_v:.3f}', ha='center', va='bottom', fontsize=9, color='#55A868')
axes[0].set_xticks(x); axes[0].set_xticklabels(targets, rotation=10, ha='right')
axes[0].set_ylabel('Error (lower is better)')
axes[0].set_title('Primary: Regression Quality')
axes[0].legend()
axes[0].set_ylim(0, max(mae_vals + rmse_vals) * 1.45)

# Right panel: binary task metrics vs NB07 baseline
models = bin_df['Model'].tolist()
x2, w2 = np.arange(len(models)), 0.25
axes[1].bar(x2 - w2, bin_df['PR-AUC'].fillna(0),  w2, label='PR-AUC',  color='#4C72B0')
axes[1].bar(x2,      bin_df['ROC-AUC'].fillna(0), w2, label='ROC-AUC', color='#DD8452')
axes[1].bar(x2 + w2, bin_df['F1'].fillna(0),      w2, label='F1',      color='#55A868')
axes[1].set_xticks(x2); axes[1].set_xticklabels(models, rotation=12, ha='right')
axes[1].set_ylim(0, 1); axes[1].set_ylabel('Score')
axes[1].set_title('Secondary: Binary Connectivity Task')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/figures/prithvi_linear_agg_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 10: Save Metrics & Push to GitHub

In [ ]:
for TARGET_NAME, m in all_metrics.items():
    pd.DataFrame([m]).to_csv(
        f'{RESULTS_DIR}/prithvi_linear_agg_{TARGET_NAME}_metrics.csv', index=False)
    # Save checkpoint path pointer
    if TARGET_NAME in best_ckpt_paths:
        with open(f'outputs/models/prithvi_linear_agg_{TARGET_NAME}_ckpt.txt', 'w') as f:
            f.write(best_ckpt_paths[TARGET_NAME])

print('Metrics saved.')

In [ ]:
import subprocess, os

token = "YOUR_TOKEN_HERE"
repo  = "tatyana21111/internet-connectivity-prediction"

subprocess.run(['git', 'config', '--global', 'user.email', 'tatyana211zy@outlook.com'], check=True)
subprocess.run(['git', 'config', '--global', 'user.name',  'tatyana21111'], check=True)
subprocess.run(['git', 'remote', 'set-url', 'origin',
                f'https://{token}@github.com/{repo}.git'], check=True)

files = [
    'outputs/results/prithvi_linear_agg_coverage_fraction_metrics.csv',
    'outputs/results/prithvi_linear_agg_log_mean_tests_metrics.csv',
    'outputs/figures/prithvi_linear_agg_coverage_fraction_val_loss.png',
    'outputs/figures/prithvi_linear_agg_coverage_fraction_regression.png',
    'outputs/figures/prithvi_linear_agg_coverage_fraction_residuals.png',
    'outputs/figures/prithvi_linear_agg_coverage_fraction_spatial.png',
    'outputs/figures/prithvi_linear_agg_coverage_fraction_binary.png',
    'outputs/figures/prithvi_linear_agg_log_mean_tests_val_loss.png',
    'outputs/figures/prithvi_linear_agg_log_mean_tests_regression.png',
    'outputs/figures/prithvi_linear_agg_log_mean_tests_residuals.png',
    'outputs/figures/prithvi_linear_agg_log_mean_tests_spatial.png',
    'outputs/figures/prithvi_linear_agg_log_mean_tests_binary.png',
    'outputs/figures/prithvi_linear_agg_comparison.png',
    'notebooks/14_prithvi_linear_agg_label.ipynb',
]
for f in files:
    if os.path.exists(f):
        subprocess.run(['git', 'add', '-f', f], check=True)
    else:
        print(f'Skipping (not yet generated): {f}')

diff = subprocess.run(['git', 'diff', '--cached', '--quiet'], capture_output=True)
if diff.returncode != 0:
    subprocess.run(['git', 'commit', '-m',
                    'Phase 1B: Prithvi-300M linear aggregate-label benchmark (NB14)'], check=True)
else:
    print('Nothing staged — all outputs may already be committed.')

subprocess.run(['git', 'pull', '--rebase', 'origin', 'main'], check=True)
subprocess.run(['git', 'push'], check=True)
print('Push successful.')